In [ ]:
print('Initializingh the project')

In [ ]:
# Install the CUDA-enabled versions of torch and torchvision specifically for this session
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [3]:
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ Using CUDA GPU")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Using Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("⚠️  Using CPU — full pipeline will be very slow. Consider Colab.")


✅ Using CUDA GPU


# 1. Kaggle and Gemini Setup

In [ ]:
# Please change these two variables in the bottom of the file `../.env `  to your Kaggle username and key, which you can find in your Kaggle account settings. 
# This will allow you to download the dataset directly from Kaggle using the Kaggle API.
# And Gemin api for RAG
# KAGGLE_USERNAME = 'your_kaggle_username'
# KAGGLE_KEY = 'your_kaggle_key'
# GEMINI_API_KEY= 'Your_gemini_api_key'


# 2. Data Download using kaggle

In [ ]:
%run ..\assignment3\dataset.py


# 3. Cleaning Data and producing these files 

- ../data/processed/counsel_chat_clean.csv   (RAG corpus)
- ../data/processed/crisis_val.csv           (~15% — validation)
- ../data/processed/crisis_train.csv         (~70% — classifier training)
- ../data/processed/crisis_test.csv          (~15% — held-out test)

In [ ]:
%run ..\assignment3\clean.py

# 4.1 Training the Model ( Optional : The already trained model is downloadable by runing Cell 4.2 instead )

In [ ]:
# %run ..\assignment3\modeling\train.py

# 4.2 Downloading the previously trained model for the project 

In [ ]:
%run ..\assignment3\download_model.py

# 5. Checking the Predictions for the crises model

In [1]:
%run ..\assignment3\modeling\predict.py

e:\poetry_cache\virtualenvs\assignment3-HMuFVmZG-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-30 09:02:34.335 | INFO     | config:<module>:17 - PROJ_ROOT path is: D:\MS_DSI\AI\Assignment3\assignment3


2026-04-30 09:02:34.366 | INFO     | __main__:main:158 - Loading model ...
2026-04-30 09:02:34.383 | INFO     | __main__:load_model:79 - Loading crisis classifier from D:\MS_DSI\AI\Assignment3\assignment3\models\crisis_classifier on cuda ...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4891.70it/s]


2026-04-30 09:02:38.736 | SUCCESS  | __main__:load_model:87 - Crisis classifier loaded.
2026-04-30 09:02:38.736 | SUCCESS  | __main__:main:160 - Model loaded.

Testing messages:
------------------------------------------------------------
✅ [NON-CRISIS] confidence: 0.05%
   I have been feeling really anxious lately and don't know what to do

🚨 [CRISIS] confidence: 98.26%
   I want to kill myself, I can't take this anymore

✅ [NON-CRISIS] confidence: 0.62%
   Can you help me with some breathing exercises?

✅ [NON-CRISIS] confidence: 4.21%
   I've been having dark thoughts and feel like ending it all

✅ [NON-CRISIS] confidence: 0.21%
   I'm struggling with my relationship and feeling lost



# 6. Evaluate the Predictions

In [ ]:
%run ..\assignment3\modeling\evaluate.py 
# if you see quota error please wait 10 min before trying again

# 7. Building the Vector Index

- 7.1 **Goal:** Persist the Counsel Chat Q&A and the curated knowledge base into a ChromaDB vector store on disk so retrieval is reproducible and fast at runtime — no need to re-embed on every app start.
- 7.2 **Two collections, one DB:** `counsel_chat` (~863 unique questions, grouped from raw rows so each unique question appears once with all therapist answers concatenated in metadata) and `knowledge_base` (~30 curated CBT / mindfulness / psychoeducation entries from `data/external/knowledge_base.json`).
- 7.3 **Embedding model:** `all-MiniLM-L6-v2` from sentence-transformers, 384-dim, with cosine similarity. The same model and space are used at retrieval time in `rag.py`, so what we build here is byte-compatible with what the runtime expects.
- 7.4 **Reproducibility:** the script `build_index.py` lives next to `config.py` and replaces what we previously did inline in this notebook. Anyone cloning the repo can rebuild the index with one command — no need to open Jupyter and run cells in order.
- 7.5 **Idempotent + selective:** by default the script skips collections that already exist. Pass `--force` to wipe and rebuild both, or `--counsel` / `--kb` to rebuild just one. Useful if you edit `knowledge_base.json` and don't want to re-embed all 863 counsel chat questions.

In [1]:
print('Creating Index- Chroma DB ')
%run ..\assignment3\build_index.py

Creating Index- Chroma DB 


e:\poetry_cache\virtualenvs\assignment3-HMuFVmZG-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-30 09:06:18.150 | INFO     | config:<module>:17 - PROJ_ROOT path is: D:\MS_DSI\AI\Assignment3\assignment3


2026-04-30 09:06:18.150 | INFO     | __main__:main:329 - Chroma DB directory: D:\MS_DSI\AI\Assignment3\assignment3\models\chroma_db
2026-04-30 09:06:18.154 | INFO     | __main__:main:330 - Embedding model:     all-MiniLM-L6-v2
2026-04-30 09:06:18.706 | INFO     | __main__:_load_embed_model:80 - Loading embedding model 'all-MiniLM-L6-v2' ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7914.22it/s]


2026-04-30 09:06:23.879 | INFO     | __main__:main:336 - === Building counsel_chat collection ===
2026-04-30 09:06:23.879 | INFO     | __main__:build_counsel_collection:145 - Loading D:\MS_DSI\AI\Assignment3\assignment3\data\processed\counsel_chat_clean.csv ...
2026-04-30 09:06:24.093 | INFO     | __main__:build_counsel_collection:148 - Loaded 2,608 raw Q&A rows
2026-04-30 09:06:24.707 | INFO     | __main__:build_counsel_collection:159 - After grouping: 863 unique questions (avg 3.0 answers each)
2026-04-30 09:06:24.894 | INFO     | __main__:build_counsel_collection:172 - Embedding 863 unique questions ...


...

2026-04-30 09:06:26.875 | INFO     | __main__:build_counsel_collection:196 - Inserting into Chroma in batches of 64 ...
2026-04-30 09:06:30.877 | SUCCESS  | __main__:build_counsel_collection:206 - Indexed 863 unique questions into 'counsel_chat'.
2026-04-30 09:06:30.877 | INFO     | __main__:main:340 - === Building knowledge_base collection ===
2026-04-30 09:06:30.891 | INFO     | __main__:_ensure_kb_json:97 - knowledge_base.json not found locally; downloading from Google Drive ...


Downloading...
From: https://drive.google.com/uc?id=17iipXI5osVQq-g060qQJinPWVF9fSVG0
To: D:\MS_DSI\AI\Assignment3\assignment3\data\external\knowledge_base.json
100%|██████████| 32.8k/32.8k [00:00<00:00, 328kB/s]


2026-04-30 09:06:33.241 | SUCCESS  | __main__:_ensure_kb_json:120 - Downloaded knowledge_base.json to D:\MS_DSI\AI\Assignment3\assignment3\data\external\knowledge_base.json
2026-04-30 09:06:33.241 | INFO     | __main__:build_kb_collection:242 - Loading D:\MS_DSI\AI\Assignment3\assignment3\data\external\knowledge_base.json ...
2026-04-30 09:06:33.264 | INFO     | __main__:build_kb_collection:245 - Loaded 30 entries from JSON
2026-04-30 09:06:33.264 | INFO     | __main__:build_kb_collection:251 -   CBT                12
2026-04-30 09:06:33.264 | INFO     | __main__:build_kb_collection:251 -   Mindfulness        8
2026-04-30 09:06:33.264 | INFO     | __main__:build_kb_collection:251 -   Psychoeducation    10
2026-04-30 09:06:33.355 | INFO     | __main__:build_kb_collection:261 - Embedding 30 entries ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]


2026-04-30 09:06:33.690 | INFO     | __main__:build_kb_collection:289 - Inserting into Chroma in batches of 64 ...
2026-04-30 09:06:34.125 | SUCCESS  | __main__:build_kb_collection:299 - Indexed 30 entries into 'knowledge_base'.
2026-04-30 09:06:34.125 | INFO     | __main__:main:344 - Final state of the Chroma DB:
2026-04-30 09:06:34.176 | INFO     | __main__:main:347 -   - counsel_chat         863 items
2026-04-30 09:06:34.176 | INFO     | __main__:main:347 -   - knowledge_base       30 items
2026-04-30 09:06:34.176 | SUCCESS  | __main__:main:348 - Index ready at D:\MS_DSI\AI\Assignment3\assignment3\models\chroma_db


# 8. Building Rag
- 8.1 Build the RAG Retrieval System: Develop the technical infrastructure to query and fetch relevant document chunks.
- 8.2 Curate Knowledge Base: Integrate Team B’s structured data into the vector store to serve as the "source of truth."
- 8.3 LLM Response Generation: Connect the Gemini API using an environment-secured key to generate natural language outputs.
- 8.4 Crisis Response Logic & Orchestration: * The Classifier: Analyzes user input for high-risk triggers.The Router: If a crisis is flagged, the system skips RAG and immediately provides empathetic support and resources.Orchestration: Wraps everything into a single respond() function that manages the logic flow: Classifier $\rightarrow$ Router $\rightarrow$ Path Selection $\rightarrow$ Return Response.

In [2]:
print('All modules imported successfully!')
%run ..\assignment3\rag.py

All modules imported successfully!

TEST 1: (RAG expected)  I've been really stressed about exams and can't focus
2026-04-30 09:06:46.256 | INFO     | __main__:get_components:256 - Initializing rag.py pipeline components ...
2026-04-30 09:06:46.256 | INFO     | __main__:get_components:259 - Loading embedding model 'all-MiniLM-L6-v2' ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4889.41it/s]


2026-04-30 09:06:51.322 | INFO     | __main__:get_components:274 - Connected to Chroma | counsel_chat: 863 | knowledge_base: 30
2026-04-30 09:06:51.322 | INFO     | __main__:get_components:285 - Loading crisis classifier on cuda ...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4589.87it/s]


2026-04-30 09:06:51.786 | INFO     | __main__:get_components:293 - Initializing Gemini client (model: gemini-2.5-flash) ...
2026-04-30 09:06:51.957 | SUCCESS  | __main__:get_components:304 - Pipeline components ready.

✅ routed to: 💬 rag
   classifier: label=non-crisis  conf=0.000  method=model

💬 REPLY:
It sounds like you're carrying a lot of stress about your exams, and that's making it hard to focus. That's a really common and understandable feeling when facing something as important as exams.

When your mind feels overwhelmed, sometimes just taking a short break to reset can be helpful. You might try something like deep breathing exercises, or even just lying down for a few minutes (S1). These kinds of small pauses can sometimes help calm your nervous system, even if it's just for a moment.

Another idea that some people find helpful is called Progressive Muscle Relaxation (PMR) (S5). This involves tensing and then relaxing different muscle groups in your body. It helps you notice 

# Perform EDA

In [ ]:
print('Performin EDA on the dataset')
%run ..\assignment3\eda.py